# Fast Priors: PNG Plots + Metrics (No PDFs)

**End-to-end, fast, PNG-only workflow** for variant prior diagnostics.

What you get:
- Predictive bands & PIT per (site, mutation) pair (PNG).
- Global summaries (PNG): μ, κ histograms; PIT histogram w/ KS p; rows-per-day; κ vs μ; process-noise plots (if available).
- Global μ(t) lines per mutation (PNG).
- Coverage-by-coverage-decile (PNG).
- **Metrics pack (CSVs only)**: predictive coverage, PIT histogram + KS p, variance ratio, LPD summary, KL info-gain summary, κ summary,
  temporal weekly medians & drift, per-group scorecard, **RMSE/MAE (counts & AF)**, calibration-by-decile, error-by-coverage-decile,
  PIT KS by site, coverage by group, etc.

No dashboards and **no PDFs** are generated. Everything is designed to be **fast**.

### Quick start
1. Set `BASE_DIR` below to your repo root **or** let the auto-detect find it.
2. Ensure `results/priors/priors_full_detail.csv` exists.
3. Run all cells.

Outputs land in:
- Figures → `results/priors/figures/*`
- Metrics → `results/priors/metric/*`

> Tip: Use the `MAX_*` and `ROW_*` limits to control runtime for very large datasets.

In [ ]:
from pathlib import Path
import os, re, warnings
warnings.filterwarnings('ignore', category=UserWarning)

# ====================== CONFIG (edit as needed) ======================
# Base directory (repo root). Leave as None to auto-detect.
BASE_DIR = None  # e.g., r"C:\\Users\\<you>\\...\\oxbio-variant-forecasting"

# Output overrides (optional). If None, defaults under BASE_DIR/results/priors
FIG_DIR_OVERRIDE = None  # e.g., r"C:\\path\\to\\results\\priors\\figures"
METRIC_DIR_OVERRIDE = None  # e.g., r"C:\\path\\to\\results\\priors\\metric"

# Predictive interval quantiles
PRED_Q_LO = 0.10
PRED_Q_HI = 0.90

# Plot settings
SAVE_PNG = True
DPI = 140
PAGESIZE = (12, 8)
PLOT_EVERY_NTH = 1          # >1 to subsample points while plotting for speed

# Limits for speed (set None for 'all')
MAX_SITES = None            # e.g., 50
MAX_MUTS  = None            # e.g., 50
MAX_PAIRS = None            # e.g., 100
ROW_LIMIT_GLOBAL = None     # e.g., 400_000 cap on heavy ops

# Metrics sampling caps (fast)
ROW_SAMPLE_PIT       = 120_000
ROW_SAMPLE_COVERAGE  = 120_000
ROW_SAMPLE_KL        = 200_000

# Metric thresholds (for scorecard)
THRESH = {
    'coverage_tol': 0.06,        # |empirical - nominal| <= 0.06
    'pit_ks_min_p': 0.05,        # KS p >= 0.05 (if SciPy present)
    'vr_min': 0.7, 'vr_max': 1.4,# variance ratio band
    'mu_drift': 3e-3,            # |slope/day| for μ
    'kappa_drift': 1e-1,         # |slope/day| for κ
    'kappa_min': 2.0, 'kappa_max': 5e3,
}
# ====================================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    'savefig.bbox': 'tight',
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    while True:
        if (p/'results'/'priors'/'priors_full_detail.csv').exists():
            return p
        if (p/'.git').exists() or (p/'configs').exists():
            root_guess = p
        if p.parent == p:
            break
        p = p.parent
    return Path(start or os.getcwd()).resolve()

REPO_ROOT = Path(BASE_DIR).resolve() if BASE_DIR else find_repo_root()
PRIORS_DIR = REPO_ROOT / 'results' / 'priors'
FIG_DIR = Path(FIG_DIR_OVERRIDE) if FIG_DIR_OVERRIDE else (PRIORS_DIR / 'fig34es')
METRIC_DIR = Path(METRIC_DIR_OVERRIDE) if METRIC_DIR_OVERRIDE else (PRIORS_DIR / 'metc')
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print(f"[info] REPO_ROOT = {REPO_ROOT}")
print(f"[info] Using priors in {PRIORS_DIR}")
print(f"[info] Figures -> {FIG_DIR}")
print(f"[info] Metrics -> {METRIC_DIR}")

In [ ]:
_INVALID = re.compile(r'[<>:"/\\|?*\x00-\x1F]')
def safe_fname(x: str, maxlen: int = 200) -> str:
    s = str(x)
    s = _INVALID.sub('_', s).replace(' ', '_').rstrip('. ').strip()
    return s[:maxlen] if maxlen else s

def _read_csv_opt(path: Path, parse_dates=None):
    return pd.read_csv(path, parse_dates=parse_dates) if path.exists() else None

CSV_PRIORS = PRIORS_DIR / 'priors_full_detail.csv'
assert CSV_PRIORS.exists(), f"Required file not found: {CSV_PRIORS}"

df = pd.read_csv(CSV_PRIORS, low_memory=False)
for c in df.columns:
    if 'date' in c.lower():
        df[c] = pd.to_datetime(df[c], errors='coerce')

# Normalize column names -> mu_t, kappa_t, count, coverage, date, mutation, site_id
lower = {c.lower(): c for c in df.columns}
mu_c    = lower.get('mu_t') or lower.get('mu')
kappa_c = lower.get('kappa_t') or lower.get('kappa')
y_c     = lower.get('count') or lower.get('y') or lower.get('success')
n_c     = lower.get('coverage') or lower.get('n') or lower.get('total')
date_c  = None
for c in df.columns:
    if 'date' in c.lower():
        date_c = c; break
mutation_c = lower.get('mutation')
site_c     = lower.get('site_id')

if mu_c is None or kappa_c is None or y_c is None or n_c is None:
    raise ValueError("priors_full_detail.csv must have mu/mu_t, kappa/kappa_t, count, coverage (and optionally date, mutation, site_id)")

# Compute AF if not present
if 'af' not in lower:
    df['af'] = np.where(df[n_c].fillna(0)>0, df[y_c]/df[n_c].clip(lower=1), np.nan)

# Downcast & sanitize
for c in [mu_c, kappa_c, y_c, n_c]:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=[mu_c, kappa_c, y_c, n_c]).copy()
df[mu_c]    = np.clip(df[mu_c].to_numpy(float), 1e-8, 1-1e-8)
df[kappa_c] = np.clip(df[kappa_c].to_numpy(float), 1e-6, np.inf)
df[y_c]     = df[y_c].astype(int)
df[n_c]     = df[n_c].astype(int)

if ROW_LIMIT_GLOBAL and len(df) > ROW_LIMIT_GLOBAL:
    df = df.sample(ROW_LIMIT_GLOBAL, random_state=42).copy()
    print(f"[warn] ROW_LIMIT_GLOBAL applied: using {len(df):,} rows")

print(df[[mu_c, kappa_c, y_c, n_c]].describe().T)

In [ ]:
try:
    from scipy.stats import betabinom, kstest
    from scipy.special import betaincinv
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _norm_ppf(p):
    p = np.asarray(p, dtype=float)
    a = [-3.969683028665376e+01, 2.209460984245205e+02,-2.759285104469687e+02,
         1.383577518672690e+02,-3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02,-1.556989798598866e+02,
         6.680131188771972e+01,-1.328068155288572e+01]
    c = [-7.784894002430293e-03,-3.223964580411365e-01,-2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,  2.445134137142996e+00,
          3.754408661907416e+00]
    plow, phigh = 0.02425, 0.97575
    x = np.empty_like(p)
    m = p < plow
    if np.any(m):
        q = np.sqrt(-2*np.log(p[m]))
        x[m] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    m = (p >= plow) & (p <= phigh)
    if np.any(m):
        q = p[m] - 0.5; r = q*q
        x[m] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
                (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)
    m = p > phigh
    if np.any(m):
        q = np.sqrt(-2*np.log(1-p[m]))
        x[m] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

def _model_mean_var(n, mu, kappa):
    a = mu*kappa; b = (1-mu)*kappa
    mean = n*mu
    # Var(Y) for Beta-Binomial
    var  = n*mu*(1-mu) * ((a+b+n)/((a+b)+1))
    return mean, np.maximum(var, 1e-12)

def _ppi_counts(n, mu, kappa, lvl=0.8):
    """Predictive interval for counts Y ~ BetaBinomial(n, mu, kappa)."""
    if _HAS_SCIPY:
        a = mu*kappa; b = (1-mu)*kappa
        alpha = (1-lvl)/2.0
        lo = betabinom.ppf(alpha, n, a, b)
        hi = betabinom.ppf(1-alpha, n, a, b)
        return lo, hi
    # normal approx w/ continuity correction
    m, v = _model_mean_var(n, mu, kappa)
    sd = np.sqrt(v)
    z  = _norm_ppf(0.5 + lvl/2.0)
    lo = np.floor(m - z*sd); hi = np.ceil(m + z*sd)
    return np.clip(lo, 0, n), np.clip(hi, 0, n)

def add_predictive_betabinom_metrics(df, mu_c, kappa_c, y_c, n_c, q_lo=0.10, q_hi=0.90):
    n  = df[n_c].to_numpy(np.int64, copy=False)
    y  = df[y_c].to_numpy(np.int64, copy=False)
    mu = np.clip(df[mu_c].to_numpy(np.float64, copy=False), 1e-9, 1-1e-9)
    k  = np.clip(df[kappa_c].to_numpy(np.float64, copy=False), 1e-8, 1e9)
    mask = n > 0
    a, b = mu*k, (1-mu)*k

    lo = np.full_like(mu, np.nan, dtype=float)
    hi = np.full_like(mu, np.nan, dtype=float)
    if _HAS_SCIPY and np.any(mask):
        p_lo = betaincinv(a[mask], b[mask], q_lo)
        p_hi = betaincinv(a[mask], b[mask], q_hi)
        lo[mask] = np.floor(n[mask] * p_lo)
        hi[mask] = np.ceil(n[mask] * p_hi)
    else:
        # normal approx around n*mu
        m, v = _model_mean_var(n, mu, k)
        sd = np.sqrt(v)
        zlo = _norm_ppf(q_lo); zhi = _norm_ppf(q_hi)
        lo = np.floor(m + zlo*sd)
        hi = np.ceil(m + zhi*sd)
        lo = np.clip(lo, 0, n); hi = np.clip(hi, 0, n)

    n_f = n.astype(float)
    df['pred_lo_ct'] = lo
    df['pred_hi_ct'] = hi
    df['pred_lo_af'] = np.divide(lo, n_f, out=np.full_like(lo, np.nan), where=mask)
    df['pred_hi_af'] = np.divide(hi, n_f, out=np.full_like(hi, np.nan), where=mask)

    # PIT (mid-P if SciPy; else normal approx)
    if _HAS_SCIPY:
        Fy  = betabinom.cdf(y, n, a, b)
        pmf = betabinom.pmf(y, n, a, b)
        u = np.clip(Fy - 0.5*pmf, 0, 1)
    else:
        m, v = _model_mean_var(n, mu, k)
        sd = np.sqrt(v)
        z = (y + 0.5 - m)/sd
        # Normal CDF approx
        u = 0.5*(1.0 + np.erf(z/np.sqrt(2)))
    u[~np.isfinite(u)] = 0.5
    df['pit_mid'] = u

    af = df.get('af', None)
    if af is None:
        af = np.where(df[n_c]>0, df[y_c]/np.maximum(df[n_c],1), np.nan)
    else:
        af = af.to_numpy(float)
    outlier = (af < df['pred_lo_af']) | (af > df['pred_hi_af'])
    df['outlier'] = np.where(np.isfinite(af), outlier, False)

    print(f"[info] Predictive bands + PIT computed for {len(df):,} rows (SciPy={_HAS_SCIPY})")
    return df

df = add_predictive_betabinom_metrics(df, mu_c, kappa_c, y_c, n_c, q_lo=PRED_Q_LO, q_hi=PRED_Q_HI)

In [ ]:
# ---------- FAST MODE KNOBS ----------
FAST_FLOAT32_AF = True          # store AF/preds as float32 to reduce bandwidth
HYBRID_INTERVAL = True          # normal approx for "easy" rows; betaincinv for the rest
RAND_PIT = False                # randomized PIT needs one extra RNG vector
CHUNK = 200_000                 # process arrays in chunks to bound SciPy overhead
EASY_MIN_NK = 40.0              # if n*kappa >= this -> normal approx is "easy"
EASY_CENTRAL_Q = (0.02, 0.98)   # central quantile range to prefer normal approx

try:
    from scipy.stats import betabinom, kstest
    from scipy.special import betaincinv
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

_rng = np.random.default_rng(42)

def _norm_ppf(p):
    p = np.asarray(p, dtype=float)
    a = [-3.969683028665376e+01, 2.209460984245205e+02,-2.759285104469687e+02,
         1.383577518672690e+02,-3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02,-1.556989798598866e+02,
         6.680131188771972e+01,-1.328068155288572e+01]
    c = [-7.784894002430293e-03,-3.223964580411365e-01,-2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,  2.445134137142996e+00,
          3.754408661907416e+00]
    plow, phigh = 0.02425, 0.97575
    x = np.empty_like(p, dtype=float)
    m = p < plow
    if np.any(m):
        q = np.sqrt(-2*np.log(p[m]))
        x[m] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    m = (p >= plow) & (p <= phigh)
    if np.any(m):
        q = p[m] - 0.5; r = q*q
        x[m] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
                (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)
    m = p > phigh
    if np.any(m):
        q = np.sqrt(-2*np.log(1-p[m]))
        x[m] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

def _model_mean_var(n, mu, kappa):
    a = mu*kappa; b = (1-mu)*kappa
    mean = n*mu
    var  = n*mu*(1-mu) * ((a+b+n)/((a+b)+1))
    return mean, np.maximum(var, 1e-12)

# Vectorized normal-approx interval used in both fallback and hybrid
def _ppi_norm(n, mu, kappa, q_lo, q_hi):
    m, v = _model_mean_var(n, mu, kappa)
    sd = np.sqrt(v)
    zlo = _norm_ppf(q_lo); zhi = _norm_ppf(q_hi)
    lo = np.floor(m + zlo*sd)
    hi = np.ceil(m + zhi*sd)
    np.clip(lo, 0, n, out=lo); np.clip(hi, 0, n, out=hi)
    return lo, hi

def _ppi_counts(n, mu, kappa, lvl=0.8, q_lo=None, q_hi=None):
    # kept for backwards-compat; prefers hybrid path below
    if q_lo is None or q_hi is None:
        alpha = (1 - lvl)/2.0
        q_lo, q_hi = alpha, 1 - alpha
    if not _HAS_SCIPY:
        return _ppi_norm(n, mu, kappa, q_lo, q_hi)
    # With SciPy, still chunk to avoid huge kernels
    N = n.shape[0]
    lo = np.empty(N, dtype=float); hi = np.empty(N, dtype=float)
    start = 0
    a = mu*kappa; b = (1-mu)*kappa
    while start < N:
        end = min(start + CHUNK, N)
        lo[start:end] = np.floor(betabinom.ppf(q_lo, n[start:end], a[start:end], b[start:end]))
        hi[start:end] = np.ceil (betabinom.ppf(q_hi, n[start:end], a[start:end], b[start:end]))
        start = end
    np.clip(lo, 0, n, out=lo); np.clip(hi, 0, n, out=hi)
    return lo, hi

def _ppi_counts_hybrid(n, mu, kappa, q_lo, q_hi):
    """Hybrid: normal approx for 'easy' rows, exact betaincinv for the rest."""
    N = n.shape[0]
    lo = np.empty(N, dtype=float); hi = np.empty(N, dtype=float)

    if not _HAS_SCIPY:
        return _ppi_norm(n, mu, kappa, q_lo, q_hi)

    nk = n * kappa
    central = (q_lo >= EASY_CENTRAL_Q[0]) and (q_hi <= EASY_CENTRAL_Q[1])
    easy_mask = (nk >= EASY_MIN_NK) & central
    hard_mask = ~easy_mask

    # Normal on easy rows
    if np.any(easy_mask):
        lo_e, hi_e = _ppi_norm(n[easy_mask], mu[easy_mask], kappa[easy_mask], q_lo, q_hi)
        lo[easy_mask] = lo_e; hi[easy_mask] = hi_e

    # Exact on hard rows (chunked betaincinv)
    idx = np.nonzero(hard_mask)[0]
    if idx.size:
        a = (mu[hard_mask] * kappa[hard_mask]).astype(float)
        b = ((1 - mu[hard_mask]) * kappa[hard_mask]).astype(float)
        nn = n[hard_mask].astype(float)

        start = 0
        while start < idx.size:
            end = min(start + CHUNK, idx.size)
            sl = slice(start, end)
            # invert on probabilities -> get p, then scale by n
            p_lo = betaincinv(a[sl], b[sl], q_lo)
            p_hi = betaincinv(a[sl], b[sl], q_hi)
            lo_h = np.floor(nn[sl] * p_lo)
            hi_h = np.ceil (nn[sl] * p_hi)
            tgt = idx[sl]
            lo[tgt] = lo_h; hi[tgt] = hi_h
            start = end

    np.clip(lo, 0, n, out=lo); np.clip(hi, 0, n, out=hi)
    return lo, hi

def add_predictive_betabinom_metrics(df, mu_c, kappa_c, y_c, n_c, q_lo=0.10, q_hi=0.90):
    # Pull arrays once; avoid copies
    n  = df[n_c].to_numpy(np.int64, copy=False)
    y  = df[y_c].to_numpy(np.int64, copy=False)
    mu = np.clip(df[mu_c].to_numpy(np.float64, copy=False), 1e-9, 1-1e-9)
    k  = np.clip(df[kappa_c].to_numpy(np.float64, copy=False), 1e-8, 1e12)
    mask = n > 0

    # --- Predictive bands (counts) ---
    if _HAS_SCIPY and HYBRID_INTERVAL:
        lo, hi = _ppi_counts_hybrid(n, mu, k, q_lo, q_hi)
    else:
        lo, hi = _ppi_counts(n, mu, k, q_hi - q_lo, q_lo, q_hi)  # reuse helper

    # --- Store AF bands; prefer float32 to reduce memory/IO
    n_f = n.astype(np.float32 if FAST_FLOAT32_AF else float, copy=False)
    pred_lo_ct = lo
    pred_hi_ct = hi
    pred_lo_af = np.full_like(n_f, np.nan)
    pred_hi_af = np.full_like(n_f, np.nan)
    np.divide(lo, n_f, out=pred_lo_af, where=mask)
    np.divide(hi, n_f, out=pred_hi_af, where=mask)

    df["pred_lo_ct"] = pred_lo_ct
    df["pred_hi_ct"] = pred_hi_ct
    df["pred_lo_af"] = pred_lo_af
    df["pred_hi_af"] = pred_hi_af

    # --- PIT (mid-P or randomized-midP), chunked when SciPy is present ---
    if _HAS_SCIPY:
        a = mu*k; b = (1 - mu)*k
        u = np.empty_like(mu)
        start = 0
        while start < n.shape[0]:
            end = min(start + CHUNK, n.shape[0])
            sl = slice(start, end)
            Fy  = betabinom.cdf(y[sl], n[sl], a[sl], b[sl])
            pmf = betabinom.pmf(y[sl], n[sl], a[sl], b[sl])
            if RAND_PIT:
                Fym1 = betabinom.cdf(np.maximum(y[sl]-1, 0), n[sl], a[sl], b[sl])
                uu = _rng.random(end - start)
                u[sl] = np.clip(Fym1 + uu*np.maximum(Fy - Fym1, 0.0), 0.0, 1.0)
            else:
                u[sl] = np.clip(Fy - 0.5*pmf, 0.0, 1.0)
            start = end
    else:
        # Fast normal approx PIT
        m, v = _model_mean_var(n, mu, k)
        sd = np.sqrt(v)
        z = (y.astype(float) + 0.5 - m)/sd
        # numpy has special.erf available as np.erf via ufunc exposure
        u = 0.5*(1.0 + np.erf(z/np.sqrt(2)))

    u[~np.isfinite(u)] = 0.5
    df["pit_mid"] = u.astype(np.float32 if FAST_FLOAT32_AF else float, copy=False)

    # --- Outlier flag (AF outside predictive AF band) ---
    if "af" in df.columns:
        af = pd.to_numeric(df["af"], errors="coerce").to_numpy(np.float32 if FAST_FLOAT32_AF else float, copy=False)
    else:
        af = np.divide(y, np.maximum(n, 1), dtype=np.float32 if FAST_FLOAT32_AF else float)
    outlier = (af < pred_lo_af) | (af > pred_hi_af)
    df["outlier"] = np.where(np.isfinite(af), outlier, False)

    print(f"[info] Predictive bands + PIT for {len(df):,} rows "
          f"(SciPy={_HAS_SCIPY}, hybrid={HYBRID_INTERVAL}, chunk={CHUNK})")
    return df
df = add_predictive_betabinom_metrics(df, mu_c, kappa_c, y_c, n_c, q_lo=PRED_Q_LO, q_hi=PRED_Q_HI)

In [ ]:
sites_all = df[site_c].dropna().astype(str).unique().tolist() if site_c in df.columns else []
muts_all  = df[mutation_c].dropna().astype(str).unique().tolist() if mutation_c in df.columns else []
sites_sel = sites_all[:MAX_SITES] if (MAX_SITES is not None) else sites_all
muts_sel  = muts_all[:MAX_MUTS]   if (MAX_MUTS is not None)   else muts_all

if site_c and mutation_c:
    df_sel = df[df[site_c].isin(sites_sel) & df[mutation_c].isin(muts_sel)].copy()
else:
    df_sel = df.copy()
print(f"[info] Selected: {len(sites_sel)}/{len(sites_all)} sites; {len(muts_sel)}/{len(muts_all)} mutations; rows={len(df_sel):,}")

In [ ]:
from matplotlib.ticker import MaxNLocator

def _ax_format_time(ax, title=None):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.set_xlabel('Date'); ax.set_ylabel('Allele frequency')
    if title: ax.set_title(title, fontsize=11)

def _plot_series_panel(ax, d, title=None):
    # Optional subsample for speed
    d_ = d.iloc[::max(1,int(PLOT_EVERY_NTH))].copy()
    ax.fill_between(d_['date'], d_['pred_lo_af'], d_['pred_hi_af'], alpha=0.25, label=f'Pred {int((PRED_Q_HI-PRED_Q_LO)*100)}% AF')
    ax.plot(d_['date'], d_['mu_t'], linewidth=2, label='μ(t)')
    # observations
    cov = d_['coverage'].fillna(0)
    sz = np.clip(np.sqrt(cov)/4.0, 3, 14)
    ax.scatter(d_['date'], d_['af'], s=sz, alpha=0.7, label='Observed AF')
    # outliers (hollow red)
    out = d_.loc[d_['outlier']]
    if not out.empty:
        ax.scatter(out['date'], out['af'], s=np.clip(np.sqrt(out['coverage'])/4.0, 3, 14), facecolors='none', edgecolors='r', linewidths=1.0, label='Outlier')
    _ax_format_time(ax, title=title)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc='best', fontsize=8, frameon=False)

def save_pair_detail_png(d, site_id, mutation, out_dir: Path):
    fig = plt.figure(figsize=PAGESIZE)
    # 2x2 grid
    gs = fig.add_gridspec(2, 2)
    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    # Top: time series
    _plot_series_panel(ax1, d, title=f"{site_id} — {mutation} (series)")

    # Bottom-left: counts vs predictive band
    d_ = d.iloc[::max(1,int(PLOT_EVERY_NTH))].copy()
    ax2.fill_between(d_['date'], d_['pred_lo_ct'], d_['pred_hi_ct'], alpha=0.25)
    ax2.plot(d_['date'], d_['count'], linewidth=1.5)
    ax2.set_title('Counts vs Predictive band'); ax2.set_ylabel('Count')
    ax2.xaxis.set_major_locator(MaxNLocator(nbins=6))

    # Bottom-right: PIT histogram
    u = d['pit_mid'].dropna().to_numpy()
    ax3.hist(u, bins=20, range=(0,1), density=True)
    ax3.set_title('PIT (mid-P)'); ax3.set_xlabel('u'); ax3.set_ylabel('density')

    fig.tight_layout()
    out = out_dir / f"pair_{safe_fname(site_id)}__{safe_fname(mutation)}.png"
    fig.savefig(out, dpi=DPI)
    plt.close(fig)
    return out


In [ ]:
pairs_dir = FIG_DIR / 'pairs'; pairs_dir.mkdir(parents=True, exist_ok=True)
glob_dir  = FIG_DIR / 'global'; glob_dir.mkdir(parents=True, exist_ok=True)
gmt_dir   = FIG_DIR / 'global_ts'; gmt_dir.mkdir(parents=True, exist_ok=True)

# --------- Per-pair PNGs ---------
pairs = df_sel[[site_c, mutation_c]].dropna().drop_duplicates().sort_values([site_c, mutation_c]).to_records(index=False) if (site_c and mutation_c) else []
if MAX_PAIRS is not None:
    pairs = pairs[:MAX_PAIRS]
count_pairs = 0
for s, m in pairs:
    d = df_sel[(df_sel[site_c]==s) & (df_sel[mutation_c]==m)].sort_values(date_c if date_c else df_sel.index.name or df_sel.index)
    if d.empty: continue
    out = save_pair_detail_png(d, s, m, pairs_dir)
    count_pairs += 1
print(f"[info] Wrote {count_pairs} per-pair PNGs -> {pairs_dir}")

# --------- Global summaries (PNG) ---------
def _save_hist(data, bins, title, xlabel, out_path):
    fig, ax = plt.subplots(figsize=(10,4))
    ax.hist(data, bins=bins)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel('count')
    fig.savefig(out_path, dpi=DPI); plt.close(fig)

_save_hist(df['mu_t'].dropna().to_numpy(float), bins=50,
           title='Distribution of μ(t)', xlabel='μ', out_path=glob_dir/'mu_distribution.png')
_save_hist(df['kappa_t'].dropna().to_numpy(float), bins=50,
           title='Distribution of κ', xlabel='κ', out_path=glob_dir/'kappa_distribution.png')

# PIT global
u = df['pit_mid'].dropna().to_numpy()
ks_p = np.nan
if u.size>0:
    if _HAS_SCIPY:
        ks_p = float(kstest(u, 'uniform').pvalue)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(u, bins=30, range=(0,1), density=True)
    ax.set_title(f'Global PIT (mid-P){'' if np.isnan(ks_p) else f' — KS p={ks_p:.3g}'}')
    ax.set_xlabel('u'); ax.set_ylabel('density')
    fig.savefig(glob_dir/'pit_hist_global.png', dpi=DPI); plt.close(fig)

# Rows per day
if date_c:
    rows_per_date = df.groupby(date_c).size().reset_index(name='n')
    fig, ax = plt.subplots(figsize=(12,4))
    ax.bar(rows_per_date[date_c], rows_per_date['n'], width=1.0)
    ax.set_title('Number of rows per day'); ax.set_ylabel('rows')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    fig.savefig(glob_dir/'rows_per_day.png', dpi=DPI); plt.close(fig)

# κ vs μ scatter
samp = df[['mu_t','kappa_t']].dropna()
if len(samp) > 200_000:
    samp = samp.sample(200_000, random_state=123)
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(samp['mu_t'], samp['kappa_t'], s=6, alpha=0.3)
ax.set_xlabel('μ'); ax.set_ylabel('κ'); ax.set_title('Scatter of κ vs μ')
fig.savefig(FIG_DIR/'scatter_kappa_vs_mu.png', dpi=DPI); plt.close(fig)

# Process noise plots if available
proc = _read_csv_opt(PRIORS_DIR / 'process_noise_by_mutation.csv')
if proc is not None and {'q_LL_hat','q_b_hat'}.issubset(proc.columns):
    x = np.clip(proc['q_LL_hat'].to_numpy(float), 1e-12, None)
    y = np.clip(proc['q_b_hat'].to_numpy(float), 1e-12, None)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(x, y, s=12, alpha=0.7)
    if np.all(x>0) and np.all(y>0):
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('q_LL_hat'); ax.set_ylabel('q_b_hat'); ax.set_title('Per-mutation process noise')
    fig.savefig(FIG_DIR/'process_noise_scatter.png', dpi=DPI); plt.close(fig)
    _save_hist(np.log10(x), bins=40, title='Distribution of log10 q_LL_hat', xlabel='log10 q_LL_hat', out_path=FIG_DIR/'q_LL_hist_log10.png')
    _save_hist(np.log10(y), bins=40, title='Distribution of log10 q_b_hat',   xlabel='log10 q_b_hat',   out_path=FIG_DIR/'q_b_hist_log10.png')

# Global μ(t) lines per mutation
gts = _read_csv_opt(PRIORS_DIR / 'detail_global_timeseries.csv', parse_dates=['date'])
if gts is not None and {'date','mutation','mu_t'}.issubset(gts.columns):
    gsrc = gts.copy()
else:
    gsrc = (df.groupby([date_c, mutation_c], as_index=False).agg(mu_t=('mu_t','mean')) if (date_c and mutation_c) else pd.DataFrame())

if not gsrc.empty:
    muts = gsrc['mutation'].dropna().astype(str).unique().tolist()
    muts = muts[:MAX_MUTS] if (MAX_MUTS is not None) else muts
    for m in muts:
        d = gsrc[gsrc['mutation']==m].sort_values('date')
        if d.empty: continue
        fig, ax = plt.subplots(figsize=(12,4))
        ax.plot(d['date'], d['mu_t'], linewidth=2)
        ax.set_ylim(-0.03, 1.03)
        ax.set_title(f'Global μ(t) — {m}')
        ax.set_xlabel('Date'); ax.set_ylabel('μ'); ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
        fig.savefig(gmt_dir / f'global_mu_{safe_fname(m)}.png', dpi=DPI); plt.close(fig)

# Coverage diagnostics by coverage decile (PNG)
cov = df[['coverage','count','pred_lo_ct','pred_hi_ct']].dropna()
cov = cov[cov['coverage']>0].copy()
if not cov.empty:
    cov['tile'] = pd.qcut(cov['coverage'], q=10, duplicates='drop')
    cov['covered'] = (cov['count'] >= cov['pred_lo_ct']) & (cov['count'] <= cov['pred_hi_ct'])
    cov_rate = cov.groupby('tile', observed=True)['covered'].mean().reset_index(name='coverage_rate')
    fig, ax = plt.subplots(figsize=(10,4))
    ax.bar(range(len(cov_rate)), cov_rate['coverage_rate'])
    ax.set_xticks(range(len(cov_rate)))
    ax.set_xticklabels(cov_rate['tile'].astype(str), rotation=45, ha='right')
    ax.set_title(f'Predictive coverage (counts) by coverage decile — target {(PRED_Q_HI-PRED_Q_LO):.0%}')
    ax.set_ylim(0, 1)
    fig.savefig(FIG_DIR / 'coverage_by_coverage_decile.png', dpi=DPI); plt.close(fig)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FAST Priors Diagnostics — Normal Approximation (CSV-only, full metrics)
Replaces Beta–Binomial predictive ops with Normal(μ·n, σ²) and writes ALL diagnostics to CSV.

Inputs:
  results/priors/priors_full_detail.csv

Outputs (all in results/priors/metric/):
  - priors_row_metrics_normalapprox.csv
  - priors_coverage_normalapprox.csv
  - priors_coverage_grid_normalapprox.csv
  - priors_pit_uniformity_bins_normalapprox.csv
  - priors_pit_tests_normalapprox.csv
  - priors_fit_diagnostics_normalapprox.csv
  - priors_variable_summaries_normalapprox.csv
  - priors_mu_kappa_correlation_normalapprox.csv
  - priors_y_mean_correlation_normalapprox.csv
  - priors_histograms_normalapprox.csv
  - priors_coverage_by_mu_quantiles_normalapprox.csv
  - priors_coverage_by_l10k_quantiles_normalapprox.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import (
    norm, kstest, chisquare, pearsonr, spearmanr, kendalltau,
    skew, kurtosis, normaltest
)
try:
    # Optional: available on SciPy >= 1.7
    from scipy.stats import cramervonmises
except Exception:
    cramervonmises = None

# ------------------- Paths -------------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
METRIC = PRIORS / "me4 w1ic"
METRIC.mkdir(parents=True, exist_ok=True)

CSV_PRIORS_FULL = PRIORS / "priors_full_detail.csv"
OUT_PREFIX = "priors"
TAG = "normalapprox"

# ------------------- Params -------------------
EPS_P = 1e-9       # for clipping probabilities
EPS_VAR = 1e-12    # to stabilize variance / sd
COVERAGE_LEVELS = (0.5, 0.9, 0.95)       # main coverage levels to report
COVERAGE_GRID = np.round(np.arange(0.50, 0.991, 0.01), 3)  # dense grid 0.50..0.99
PIT_BINS = 20
BINS_HIST = 60

# ------------------- Helpers -------------------
def _save_csv(df, name):
    p = METRIC / f"{name}.csv"
    df.to_csv(p, index=False, float_format="%.6g")
    print("[csv]", p)

def _clip01(x, eps=EPS_P):
    return np.clip(x, eps, 1 - eps)

def ab_from_mu_kappa(mu, kappa):
    mu = _clip01(mu)
    kappa = np.clip(kappa, 1e-6, np.inf)
    return mu * kappa, (1 - mu) * kappa

def _hist_df(x, bins=50, rng=None, variable="value"):
    counts, edges = np.histogram(x, bins=bins, range=rng)
    widths = edges[1:] - edges[:-1]
    total = counts.sum()
    density = np.zeros_like(counts, dtype=float)
    if total > 0:
        density = counts / total
        # Convert to density per unit-width so area integrates to 1:
        with np.errstate(divide="ignore", invalid="ignore"):
            density = np.where(widths > 0, density / widths, 0.0)
    return pd.DataFrame({
        "variable": variable,
        "bin_left": edges[:-1],
        "bin_right": edges[1:],
        "count": counts,
        "density": density
    })

def _summarize(varname, arr):
    s = pd.Series(arr).describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    s = s.rename(index={
        "count":"count","mean":"mean","std":"std","min":"min","1%":"q01",
        "5%":"q05","10%":"q10","25%":"q25","50%":"q50","75%":"q75",
        "90%":"q90","95%":"q95","99%":"q99","max":"max"
    })
    df = s.to_frame("value").reset_index().rename(columns={"index":"stat"})
    df.insert(0, "variable", varname)
    return df

def _covariance(x, y):
    # population covariance (ddof=0), numeric stable
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xm = np.mean(x)
    ym = np.mean(y)
    return np.mean((x - xm) * (y - ym))

def _pearson(x, y):
    r, p = pearsonr(x, y) if hasattr(pearsonr(x, y), "__iter__") else (pearsonr(x,y).statistic, pearsonr(x,y).pvalue)
    # SciPy < 1.10 returns tuple; >=1.10 returns result object; robustly handle both:
    try:
        res = pearsonr(x, y)
        r = res.statistic if hasattr(res, "statistic") else res[0]
        p = res.pvalue if hasattr(res, "pvalue") else res[1]
    except Exception:
        pass
    return float(r), float(p)

def _spearman(x, y):
    res = spearmanr(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue if hasattr(res, "pvalue") else res[1]
    return float(r), float(p)

def _kendall(x, y):
    res = kendalltau(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue if hasattr(res, "pvalue") else res[1]
    return float(r), float(p)

def _safe_normaltest(x):
    # D'Agostino K^2 test for normality; with very large N it will be sensitive.
    try:
        res = normaltest(x, nan_policy="omit")
        stat = float(res.statistic if hasattr(res, "statistic") else res[0])
        pval = float(res.pvalue if hasattr(res, "pvalue") else res[1])
        return stat, pval
    except Exception:
        return np.nan, np.nan

def _fname(stem):
    return f"{OUT_PREFIX}_{stem}_{TAG}"

# ------------------- Load -------------------
df = pd.read_csv(CSV_PRIORS_FULL, low_memory=False)
df.rename(columns={"mu_t": "mu", "kappa_t": "kappa"}, inplace=True)

for c in ["mu", "kappa", "count", "coverage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["mu", "kappa", "count", "coverage"])
df["count"] = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)

# Basic guards
df = df[df["coverage"] > 0]
df["count"] = np.clip(df["count"], 0, df["coverage"])
df["mu"] = _clip01(df["mu"])
df["kappa"] = np.clip(df["kappa"], 1e-6, np.inf)

print(f"Loaded {len(df):,} rows")

# ------------------- Arrays -------------------
y = df["count"].to_numpy()
n = df["coverage"].to_numpy()
mu = df["mu"].to_numpy()
kappa = df["kappa"].to_numpy()
a, b = ab_from_mu_kappa(mu, kappa)

# Normal approximation parameters for Beta–Binomial predictive
mean = n * mu
var = n * mu * (1 - mu) * ((a + b + n) / (a + b + 1))
var = np.maximum(var, EPS_VAR)
sd = np.sqrt(var)

# Core per-row diagnostics
resid = y - mean
resid2 = resid ** 2
z = (y - mean) / sd
U = norm.cdf(z)  # PIT under Normal approx
with np.errstate(divide="ignore"):
    logpdf = norm.logpdf(y, mean, sd)

vr = resid2 / var

# ------------------- 1) Per-row metrics (wide, detailed) -------------------
# Add common credible interval limits & in/out flags
def _ci_flags(level):
    alpha = (1.0 - level) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    return lo, hi, inside

lo50, hi50, in50 = _ci_flags(0.50)
lo90, hi90, in90 = _ci_flags(0.90)
lo95, hi95, in95 = _ci_flags(0.95)

row_metrics = pd.DataFrame({
    "row_index": np.arange(len(df), dtype=int),
    "y": y,
    "n": n,
    "mu": mu,
    "kappa": kappa,
    "mean": mean,
    "var": var,
    "sd": sd,
    "z": z,
    "u_pit": U,
    "logpdf": logpdf,
    "resid": resid,
    "resid2": resid2,
    "variance_ratio": vr,
    "lo_50": lo50, "hi_50": hi50, "in_50": in50.astype(int),
    "lo_90": lo90, "hi_90": hi90, "in_90": in90.astype(int),
    "lo_95": lo95, "hi_95": hi95, "in_95": in95.astype(int),
})
_save_csv(row_metrics, _fname("row_metrics"))

# ------------------- 2) Coverage calibration (requested levels) ------------
cov_rows = []
for lvl in COVERAGE_LEVELS:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    lower_miss = (y < lo)
    upper_miss = (y > hi)
    cov_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "lower_miss_rate": float(np.mean(lower_miss)),
        "upper_miss_rate": float(np.mean(upper_miss)),
        "mean_width": float(np.mean(hi - lo)),
        "median_width": float(np.median(hi - lo)),
        "sample_n": int(len(y)),
    })
cov_tbl = pd.DataFrame(cov_rows)
_save_csv(cov_tbl, _fname("coverage"))

# Dense coverage grid
cov_grid_rows = []
for lvl in COVERAGE_GRID:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    cov_grid_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "sample_n": int(len(y))
    })
cov_grid = pd.DataFrame(cov_grid_rows)
_save_csv(cov_grid, _fname("coverage_grid"))

# ------------------- 3) PIT uniformity (bins + tests) ----------------------
# Histogram
pit_hist, pit_edges = np.histogram(U, bins=PIT_BINS, range=(0, 1))
expected = len(U) / PIT_BINS if PIT_BINS > 0 else np.nan
pit_tbl = pd.DataFrame({
    "bin_left": pit_edges[:-1],
    "bin_right": pit_edges[1:],
    "count": pit_hist,
    "density": pit_hist / pit_hist.sum() if pit_hist.sum() > 0 else np.zeros_like(pit_hist),
    "expected_count_uniform": expected
})
_save_csv(pit_tbl, _fname("pit_uniformity_bins"))

# Tests
ks = kstest(U, "uniform")
ks_stat = float(getattr(ks, "statistic", ks[0]))
ks_pval = float(getattr(ks, "pvalue", ks[1]))

chi = chisquare(f_obs=pit_hist, f_exp=np.full_like(pit_hist, expected, dtype=float)) if expected > 0 else None
chi_stat = float(getattr(chi, "statistic", np.nan)) if chi is not None else np.nan
chi_pval = float(getattr(chi, "pvalue", np.nan)) if chi is not None else np.nan

cvm_stat, cvm_pval = np.nan, np.nan
if cramervonmises is not None:
    try:
        cvm = cramervonmises(U, "uniform")
        cvm_stat = float(getattr(cvm, "statistic", np.nan))
        cvm_pval = float(getattr(cvm, "pvalue", np.nan))
    except Exception:
        pass

pit_tests = pd.DataFrame([{
    "test": "KS (Uniform)",
    "statistic": ks_stat,
    "pvalue": ks_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}, {
    "test": "Chi-square (Uniform bins)",
    "statistic": chi_stat,
    "pvalue": chi_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}, {
    "test": "Cramér–von Mises (Uniform)",
    "statistic": cvm_stat,
    "pvalue": cvm_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}])
_save_csv(pit_tests, _fname("pit_tests"))

# ------------------- 4) Global fit diagnostics -----------------------------
rmse = float(np.sqrt(np.mean(resid2)))
mae = float(np.mean(np.abs(resid)))
rmsz = float(np.sqrt(np.mean(z**2)))
z_mean = float(np.mean(z))
z_std = float(np.std(z))
z_skew = float(skew(z, nan_policy="omit"))
z_kurt = float(kurtosis(z, fisher=True, nan_policy="omit"))
k2_stat, k2_p = _safe_normaltest(z)

vr_mean = float(np.mean(vr[np.isfinite(vr)]))
vr_med  = float(np.median(vr[np.isfinite(vr)]))
vr_q05  = float(np.quantile(vr[np.isfinite(vr)], 0.05))
vr_q95  = float(np.quantile(vr[np.isfinite(vr)], 0.95))

# y vs predicted mean correlation and simple regression
r_ym, p_ym = _pearson(y, mean)
rs_ym, ps_ym = _spearman(y, mean)
cov_ym = _covariance(mean, y)
var_m = float(np.var(mean))
if var_m > 0:
    slope = float(cov_ym / var_m)
    intercept = float(np.mean(y) - slope * np.mean(mean))
else:
    slope, intercept = np.nan, np.nan
r2 = float(r_ym**2)

lpd_mean = float(np.mean(logpdf[np.isfinite(logpdf)]))

fit_diag = pd.DataFrame([{
    "N": int(len(y)),
    "rmse": rmse,
    "mae": mae,
    "rmsz": rmsz,
    "avg_logpdf_normal": lpd_mean,
    "variance_ratio_mean": vr_mean,
    "variance_ratio_median": vr_med,
    "variance_ratio_q05": vr_q05,
    "variance_ratio_q95": vr_q95,
    "z_mean": z_mean,
    "z_std": z_std,
    "z_skew": z_skew,
    "z_kurtosis_fisher": z_kurt,
    "normaltest_k2_stat": k2_stat,
    "normaltest_k2_pvalue": k2_p,
    "pearson_y_mean_r": r_ym,
    "pearson_y_mean_p": p_ym,
    "spearman_y_mean_rho": rs_ym,
    "spearman_y_mean_p": ps_ym,
    "regress_y_on_mean_slope": slope,
    "regress_y_on_mean_intercept": intercept,
    "regress_y_on_mean_r2": r2
}])
_save_csv(fit_diag, _fname("fit_diagnostics"))

# ------------------- 5) κ, μ, and other summaries --------------------------
summaries = pd.concat([
    _summarize("y", y),
    _summarize("n", n),
    _summarize("mu", mu),
    _summarize("kappa", kappa),
    _summarize("mean", mean),
    _summarize("var", var),
    _summarize("sd", sd),
    _summarize("z", z),
    _summarize("u_pit", U),
    _summarize("resid", resid),
    _summarize("resid2", resid2),
    _summarize("variance_ratio", vr),
    _summarize("logpdf", logpdf[np.isfinite(logpdf)])
], ignore_index=True)
_save_csv(summaries, _fname("variable_summaries"))

# Correlations μ–κ
r_pk, p_pk = _pearson(mu, kappa)
r_sk, p_sk = _spearman(mu, kappa)
r_kk, p_kk = _kendall(mu, kappa)
mu_kappa_corr = pd.DataFrame([{
    "pearson_mu_kappa": r_pk, "pearson_p": p_pk,
    "spearman_mu_kappa": r_sk, "spearman_p": p_sk,
    "kendall_mu_kappa": r_kk, "kendall_p": p_kk
}])
_save_csv(mu_kappa_corr, _fname("mu_kappa_correlation"))

# y–mean correlations already saved; also export separately for convenience
y_mean_corr = pd.DataFrame([{
    "pearson_y_mean": r_ym, "pearson_p": p_ym,
    "spearman_y_mean": rs_ym, "spearman_p": ps_ym,
    "slope_y_on_mean": slope, "intercept_y_on_mean": intercept, "r2": r2
}])
_save_csv(y_mean_corr, _fname("y_mean_correlation"))

# ------------------- 6) Histograms (tidy) ----------------------------------
l10k = np.log10(np.clip(kappa, 1e-12, None))
hist_mu    = _hist_df(mu,   bins=BINS_HIST, rng=(0.0, 1.0), variable="mu")
hist_l10k  = _hist_df(l10k, bins=BINS_HIST, rng=(float(np.min(l10k)), float(np.max(l10k))), variable="log10_kappa")
# Use robust range for z (clip to 0.1%..99.9% quantiles to avoid infinite axis in densities)
z_lo, z_hi = float(np.quantile(z, 0.001)), float(np.quantile(z, 0.999))
hist_z     = _hist_df(np.clip(z, z_lo, z_hi), bins=BINS_HIST, rng=(z_lo, z_hi), variable="z")
hist_u     = _hist_df(U,    bins=BINS_HIST, rng=(0.0, 1.0), variable="u_pit")
# VR can be heavy-tailed; cap at 99.5% for histogram range
vr_lo, vr_hi = float(np.quantile(vr[np.isfinite(vr)], 0.0)), float(np.quantile(vr[np.isfinite(vr)], 0.995))
hist_vr    = _hist_df(np.clip(vr, vr_lo, vr_hi), bins=BINS_HIST, rng=(vr_lo, vr_hi), variable="variance_ratio")

hists = pd.concat([hist_mu, hist_l10k, hist_z, hist_u, hist_vr], ignore_index=True)
_save_csv(hists, _fname("histograms"))

# ------------------- 7) Calibration by μ/κ deciles -------------------------
def _coverage_by_quantiles(key, values, levels=COVERAGE_LEVELS, q=10):
    labels = pd.qcut(values, q=q, duplicates="drop")
    dfq = pd.DataFrame({key: values, "label": labels})
    # Build indexes to speed mask application
    group = dfq["label"].astype(str)
    bounds = dfq["label"].cat.categories if hasattr(dfq["label"], "cat") else sorted(group.unique())
    rows = []
    # Precompute CIs for each level
    cis = {}
    for lvl in levels:
        alpha = (1.0 - lvl) / 2.0
        lo = norm.ppf(alpha, loc=mean, scale=sd)
        hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
        cis[lvl] = (lo, hi)
    # Iterate groups
    for lbl in group.unique():
        mask = (group == lbl).to_numpy()
        if mask.sum() == 0:
            continue
        key_vals = values[mask]
        mu_mean = float(np.mean(mu[mask]))
        k_mean  = float(np.mean(kappa[mask]))
        # Compute per level
        for lvl in levels:
            lo, hi = cis[lvl]
            inside = (y >= lo) & (y <= hi)
            emp = float(np.mean(inside[mask]))
            rows.append({
                "bin": str(lbl),
                "nominal": float(lvl),
                "empirical": emp,
                "bias": emp - float(lvl),
                "N_bin": int(mask.sum()),
                "mu_mean": mu_mean,
                "kappa_mean": k_mean
            })
    return pd.DataFrame(rows)

cov_by_mu    = _coverage_by_quantiles("mu",   mu,   levels=COVERAGE_LEVELS, q=10)
cov_by_l10k  = _coverage_by_quantiles("l10k", l10k, levels=COVERAGE_LEVELS, q=10)

_save_csv(cov_by_mu,   _fname("coverage_by_mu_quantiles"))
_save_csv(cov_by_l10k, _fname("coverage_by_l10k_quantiles"))

print("\n✅ Normal-approx priors diagnostics complete. All outputs written as CSVs in 'metric'.")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FAST Priors Diagnostics — Normal Approximation (CSV-only, full metrics + summary)
"""

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import (
    norm, kstest, chisquare, pearsonr, spearmanr, kendalltau,
    skew, kurtosis, normaltest
)
try:
    from scipy.stats import cramervonmises  # SciPy >= 1.7
except Exception:
    cramervonmises = None

# ------------------- Paths -------------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
METRIC = PRIORS / "me4c87n3ciuhehc34cy43finaltreyw1ic"
METRIC.mkdir(parents=True, exist_ok=True)
print(f"[outdir] {METRIC}")

CSV_PRIORS_FULL = PRIORS / "priors_full_detail.csv"
OUT_PREFIX = "priors"
TAG = "normalapprox"

# ------------------- Params -------------------
EPS_P   = 1e-9
EPS_VAR = 1e-12
COVERAGE_LEVELS = (0.50, 0.90, 0.95)
COVERAGE_GRID   = np.round(np.arange(0.50, 0.991, 0.01), 3)
PIT_BINS  = 20
BINS_HIST = 60

# ------------------- Helpers -------------------
def _save_csv(df, name):
    p = METRIC / f"{name}.csv"
    df.to_csv(p, index=False, float_format="%.6g")
    print("[csv]", p)

def _clip01(x, eps=EPS_P):
    return np.clip(x, eps, 1 - eps)

def ab_from_mu_kappa(mu, kappa):
    mu = _clip01(mu); kappa = np.clip(kappa, 1e-6, np.inf)
    return mu * kappa, (1 - mu) * kappa

def _safe_range(lo, hi, eps=1e-9):
    if not np.isfinite(lo) or not np.isfinite(hi):
        return (-1.0, 1.0)
    if hi - lo < eps:
        c = 0.5 * (lo + hi); w = max(abs(c), 1.0)
        return (c - w, c + w)
    return (float(lo), float(hi))

def _hist_df(x, bins=50, rng=None, variable="value"):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return pd.DataFrame(columns=["variable","bin_left","bin_right","count","density"])
    if rng is None:
        lo, hi = np.nanmin(x), np.nanmax(x); rng = _safe_range(lo, hi)
    counts, edges = np.histogram(x, bins=bins, range=rng)
    widths = edges[1:] - edges[:-1]
    total = counts.sum()
    density = np.zeros_like(counts, dtype=float)
    if total > 0:
        pmf = counts / total
        with np.errstate(divide="ignore", invalid="ignore"):
            density = np.where(widths > 0, pmf / widths, 0.0)
    return pd.DataFrame({
        "variable": variable,
        "bin_left": edges[:-1],
        "bin_right": edges[1:],
        "count": counts,
        "density": density
    })

def _summarize(varname, arr):
    arr = np.asarray(arr, float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return pd.DataFrame({"variable":[varname],"stat":["count"],"value":[0.0]})
    s = pd.Series(arr).describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    s = s.rename(index={
        "count":"count","mean":"mean","std":"std","min":"min","1%":"q01",
        "5%":"q05","10%":"q10","25%":"q25","50%":"q50","75%":"q75",
        "90%":"q90","95%":"q95","99%":"q99","max":"max"
    })
    df = s.to_frame("value").reset_index().rename(columns={"index":"stat"})
    df.insert(0, "variable", varname)
    return df

def _covariance(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    xm = np.mean(x); ym = np.mean(y)
    return np.mean((x - xm) * (y - ym))

def _pearson(x, y):
    res = pearsonr(x, y); 
    return float(getattr(res,"statistic",res[0])), float(getattr(res,"pvalue",res[1]))

def _spearman(x, y):
    res = spearmanr(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue      if hasattr(res, "pvalue")      else res[1]
    return float(r), float(p)

def _kendall(x, y):
    res = kendalltau(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue      if hasattr(res, "pvalue")      else res[1]
    return float(r), float(p)

def _safe_normaltest(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan
    try:
        res = normaltest(x, nan_policy="omit")
        return float(getattr(res,"statistic",res[0])), float(getattr(res,"pvalue",res[1]))
    except Exception:
        return np.nan, np.nan

def _fname(stem): return f"{OUT_PREFIX}_{stem}_{TAG}"

# ------------------- Load (column-flexible) -------------------
df = pd.read_csv(CSV_PRIORS_FULL, low_memory=False)

# Map columns flexibly:
colmap = {}
# mu
if "mu" in df.columns: colmap["mu"] = "mu"
elif "mu_t" in df.columns: colmap["mu"] = "mu_t"
else: raise ValueError("Could not find 'mu' or 'mu_t' in priors_full_detail.csv")

# kappa
if "kappa" in df.columns: colmap["kappa"] = "kappa"
elif "kappa_t" in df.columns: colmap["kappa"] = "kappa_t"
else: raise ValueError("Could not find 'kappa' or 'kappa_t' in priors_full_detail.csv")

# counts
if "count" in df.columns: colmap["count"] = "count"
elif "y" in df.columns: colmap["count"] = "y"
else: raise ValueError("Could not find 'count' or 'y' in priors_full_detail.csv")

# coverage
if "coverage" in df.columns: colmap["coverage"] = "coverage"
elif "n" in df.columns: colmap["coverage"] = "n"
else: raise ValueError("Could not find 'coverage' or 'n' in priors_full_detail.csv")

df = df.rename(columns={
    colmap["mu"]: "mu",
    colmap["kappa"]: "kappa",
    colmap["count"]: "count",
    colmap["coverage"]: "coverage",
})

for c in ["mu","kappa","count","coverage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mu","kappa","count","coverage"])
df["count"] = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)

# Guards
df = df[df["coverage"] > 0]
df["count"] = np.clip(df["count"], 0, df["coverage"])
df["mu"]    = _clip01(df["mu"])
df["kappa"] = np.clip(df["kappa"], 1e-6, np.inf)

print(f"[loaded] {len(df):,} rows")

# ------------------- Arrays -------------------
y = df["count"].to_numpy()
n = df["coverage"].to_numpy()
mu = df["mu"].to_numpy()
kappa = df["kappa"].to_numpy()
a, b = ab_from_mu_kappa(mu, kappa)

# Normal-approx predictive
mean = n * mu
# Var[BB] = n * μ (1-μ) * ((a+b+n)/(a+b+1))
var = n * mu * (1 - mu) * ((a + b + n) / (a + b + 1))
var = np.maximum(var, EPS_VAR)
sd  = np.sqrt(var)

# Per-row diagnostics
resid  = y - mean
resid2 = resid ** 2
z = (y - mean) / sd
U = norm.cdf(z)
with np.errstate(divide="ignore", invalid="ignore"):
    logpdf = norm.logpdf(y, mean, sd)
vr = resid2 / var

# ------------------- 1) Per-row metrics -------------------
def _ci_flags(level):
    alpha = (1.0 - level) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    return lo, hi, inside

lo50, hi50, in50 = _ci_flags(0.50)
lo90, hi90, in90 = _ci_flags(0.90)
lo95, hi95, in95 = _ci_flags(0.95)

row_metrics = pd.DataFrame({
    "row_index": np.arange(len(df), dtype=int),
    "y": y, "n": n, "mu": mu, "kappa": kappa,
    "mean": mean, "var": var, "sd": sd,
    "z": z, "u_pit": U, "logpdf": logpdf,
    "resid": resid, "resid2": resid2, "variance_ratio": vr,
    "lo_50": lo50, "hi_50": hi50, "in_50": in50.astype(int),
    "lo_90": lo90, "hi_90": hi90, "in_90": in90.astype(int),
    "lo_95": lo95, "hi_95": hi95, "in_95": in95.astype(int),
})
_save_csv(row_metrics, _fname("row_metrics"))

# ------------------- 2) Coverage (levels + grid) -------------------
cov_rows = []
for lvl in COVERAGE_LEVELS:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    lower_miss = (y < lo)
    upper_miss = (y > hi)
    cov_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "lower_miss_rate": float(np.mean(lower_miss)),
        "upper_miss_rate": float(np.mean(upper_miss)),
        "mean_width": float(np.mean(hi - lo)),
        "median_width": float(np.median(hi - lo)),
        "sample_n": int(len(y)),
    })
cov_tbl = pd.DataFrame(cov_rows)
_save_csv(cov_tbl, _fname("coverage"))

cov_grid_rows = []
for lvl in COVERAGE_GRID:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    cov_grid_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "sample_n": int(len(y))
    })
cov_grid = pd.DataFrame(cov_grid_rows)
_save_csv(cov_grid, _fname("coverage_grid"))

# ------------------- 3) PIT uniformity -------------------
pit_hist, pit_edges = np.histogram(U, bins=PIT_BINS, range=(0, 1))
expected = len(U) / PIT_BINS if PIT_BINS > 0 else np.nan
pit_tbl = pd.DataFrame({
    "bin_left": pit_edges[:-1],
    "bin_right": pit_edges[1:],
    "count": pit_hist,
    "density": pit_hist / pit_hist.sum() if pit_hist.sum() > 0 else np.zeros_like(pit_hist),
    "expected_count_uniform": expected
})
_save_csv(pit_tbl, _fname("pit_uniformity_bins"))

ks = kstest(U, "uniform")
ks_stat = float(getattr(ks, "statistic", ks[0] if isinstance(ks, tuple) else np.nan))
ks_pval = float(getattr(ks, "pvalue",   ks[1] if isinstance(ks, tuple) else np.nan))

chi = chisquare(f_obs=pit_hist, f_exp=np.full_like(pit_hist, expected, dtype=float)) if expected > 0 else None
chi_stat = float(getattr(chi, "statistic", np.nan)) if chi is not None else np.nan
chi_pval = float(getattr(chi, "pvalue",   np.nan))  if chi is not None else np.nan

cvm_stat, cvm_pval = np.nan, np.nan
if cramervonmises is not None:
    try:
        cvm = cramervonmises(U, "uniform")
        cvm_stat = float(getattr(cvm, "statistic", np.nan))
        cvm_pval = float(getattr(cvm, "pvalue",   np.nan))
    except Exception:
        pass

pit_tests = pd.DataFrame([{
    "test": "KS (Uniform)", "statistic": ks_stat, "pvalue": ks_pval, "N": int(len(U)), "bins": int(PIT_BINS)
}, {
    "test": "Chi-square (Uniform bins)", "statistic": chi_stat, "pvalue": chi_pval, "N": int(len(U)), "bins": int(PIT_BINS)
}, {
    "test": "Cramér–von Mises (Uniform)", "statistic": cvm_stat, "pvalue": cvm_pval, "N": int(len(U)), "bins": int(PIT_BINS)
}])
_save_csv(pit_tests, _fname("pit_tests"))

# ------------------- 4) Global fit diagnostics -------------------
rmse   = float(np.sqrt(np.mean(resid2)))
mae    = float(np.mean(np.abs(resid)))
rmsz   = float(np.sqrt(np.mean(z**2)))
z_mean = float(np.mean(z))
z_std  = float(np.std(z))
z_skew = float(skew(z, nan_policy="omit"))
z_kurt = float(kurtosis(z, fisher=True, nan_policy="omit"))
k2_stat, k2_p = _safe_normaltest(z)

vr_finite = vr[np.isfinite(vr)]
vr_mean = float(np.mean(vr_finite)) if vr_finite.size else np.nan
vr_med  = float(np.median(vr_finite)) if vr_finite.size else np.nan
vr_q05  = float(np.quantile(vr_finite, 0.05)) if vr_finite.size else np.nan
vr_q95  = float(np.quantile(vr_finite, 0.95)) if vr_finite.size else np.nan

r_ym, p_ym   = _pearson(y, mean)
rs_ym, ps_ym = _spearman(y, mean)
cov_ym = _covariance(mean, y)
var_m  = float(np.var(mean))
if var_m > 0:
    slope = float(cov_ym / var_m)
    intercept = float(np.mean(y) - slope * np.mean(mean))
else:
    slope, intercept = np.nan, np.nan
r2 = float(r_ym**2)

with np.errstate(invalid="ignore"):
    lpd_mean = float(np.nanmean(logpdf))

fit_diag = pd.DataFrame([{
    "N": int(len(y)),
    "rmse": rmse, "mae": mae, "rmsz": rmsz, "avg_logpdf_normal": lpd_mean,
    "variance_ratio_mean": vr_mean, "variance_ratio_median": vr_med,
    "variance_ratio_q05": vr_q05, "variance_ratio_q95": vr_q95,
    "z_mean": z_mean, "z_std": z_std, "z_skew": z_skew, "z_kurtosis_fisher": z_kurt,
    "normaltest_k2_stat": k2_stat, "normaltest_k2_pvalue": k2_p,
    "pearson_y_mean_r": r_ym, "pearson_y_mean_p": p_ym,
    "spearman_y_mean_rho": rs_ym, "spearman_y_mean_p": ps_ym,
    "regress_y_on_mean_slope": slope, "regress_y_on_mean_intercept": intercept,
    "regress_y_on_mean_r2": r2
}])
_save_csv(fit_diag, _fname("fit_diagnostics"))

# ------------------- 4b) Compact overall summary -------------------
def _cov_pick(tbl, level):
    row = tbl.loc[np.isclose(tbl["nominal"], level)]
    if len(row) == 0: return np.nan, np.nan
    row = row.iloc[0]; return float(row["empirical"]), float(row["bias"])

cov50_emp, cov50_bias = _cov_pick(cov_tbl, 0.50)
cov90_emp, cov90_bias = _cov_pick(cov_tbl, 0.90)
cov95_emp, cov95_bias = _cov_pick(cov_tbl, 0.95)

def _pit_pick(name):
    row = pit_tests.loc[pit_tests["test"] == name]
    return float(row.iloc[0]["pvalue"]) if len(row) else np.nan

ks_p  = _pit_pick("KS (Uniform)")
chi_p = _pit_pick("Chi-square (Uniform bins)")
cvm_p = _pit_pick("Cramér–von Mises (Uniform)")

summary = pd.DataFrame([{
    "N": int(len(y)),
    "rmse": rmse, "mae": mae, "rmsz": rmsz,
    "z_mean": z_mean, "z_std": z_std, "z_skew": z_skew, "z_kurtosis_fisher": z_kurt,
    "avg_logpdf_normal": lpd_mean,
    "pearson_y_mean_r": r_ym, "spearman_y_mean_rho": rs_ym,
    "regress_y_on_mean_slope": slope, "regress_y_on_mean_intercept": intercept, "regress_y_on_mean_r2": r2,
    "variance_ratio_mean": vr_mean, "variance_ratio_median": vr_med,
    "variance_ratio_q05": vr_q05, "variance_ratio_q95": vr_q95,
    "cov50_empirical": cov50_emp, "cov50_bias": cov50_bias,
    "cov90_empirical": cov90_emp, "cov90_bias": cov90_bias,
    "cov95_empirical": cov95_emp, "cov95_bias": cov95_bias,
    "pit_ks_pvalue": ks_p, "pit_chisq_pvalue": chi_p, "pit_cvm_pvalue": cvm_p,
}])
_save_csv(summary, _fname("summary"))
print(f"✅ Summary written -> {(METRIC / (_fname('summary') + '.csv'))}")

# ------------------- 5) κ, μ, and other summaries -------------------
summaries = pd.concat([
    _summarize("y", y), _summarize("n", n), _summarize("mu", mu), _summarize("kappa", kappa),
    _summarize("mean", mean), _summarize("var", var), _summarize("sd", sd),
    _summarize("z", z), _summarize("u_pit", U), _summarize("resid", resid),
    _summarize("resid2", resid2), _summarize("variance_ratio", vr),
    _summarize("logpdf", logpdf[np.isfinite(logpdf)])
], ignore_index=True)
_save_csv(summaries, _fname("variable_summaries"))

# μ–κ correlations
r_pk, p_pk = _pearson(mu, kappa)
r_sk, p_sk = _spearman(mu, kappa)
r_kk, p_kk = _kendall(mu, kappa)
mu_kappa_corr = pd.DataFrame([{
    "pearson_mu_kappa": r_pk, "pearson_p": p_pk,
    "spearman_mu_kappa": r_sk, "spearman_p": p_sk,
    "kendall_mu_kappa": r_kk, "kendall_p": p_kk
}])
_save_csv(mu_kappa_corr, _fname("mu_kappa_correlation"))

# y–mean correlations (convenience)
y_mean_corr = pd.DataFrame([{
    "pearson_y_mean": r_ym, "pearson_p": p_ym,
    "spearman_y_mean": rs_ym, "spearman_p": ps_ym,
    "slope_y_on_mean": slope, "intercept_y_on_mean": intercept, "r2": r2
}])
_save_csv(y_mean_corr, _fname("y_mean_correlation"))

# ------------------- 6) Histograms (tidy) -------------------
l10k = np.log10(np.clip(kappa, 1e-12, None))
mu_rng   = (0.0, 1.0)
l10k_rng = _safe_range(float(np.min(l10k)), float(np.max(l10k)))
z_lo, z_hi = float(np.quantile(z, 0.001)), float(np.quantile(z, 0.999))
z_rng = _safe_range(z_lo, z_hi)
if vr_finite.size:
    vr_lo, vr_hi = float(np.quantile(vr_finite, 0.00)), float(np.quantile(vr_finite, 0.995))
else:
    vr_lo, vr_hi = 0.0, 1.0
vr_rng = _safe_range(vr_lo, vr_hi)

hist_mu   = _hist_df(mu,   bins=BINS_HIST, rng=mu_rng,   variable="mu")
hist_l10k = _hist_df(l10k, bins=BINS_HIST, rng=l10k_rng, variable="log10_kappa")
hist_z    = _hist_df(np.clip(z, z_rng[0], z_rng[1]), bins=BINS_HIST, rng=z_rng, variable="z")
hist_u    = _hist_df(U,    bins=BINS_HIST, rng=(0.0, 1.0), variable="u_pit")
hist_vr   = _hist_df(np.clip(vr, vr_rng[0], vr_rng[1]), bins=BINS_HIST, rng=vr_rng, variable="variance_ratio")

hists = pd.concat([hist_mu, hist_l10k, hist_z, hist_u, hist_vr], ignore_index=True)
_save_csv(hists, _fname("histograms"))
# ------------------- 7) Calibration by μ/κ deciles -------------------------
def _coverage_by_quantiles(values, levels=COVERAGE_LEVELS, q=10):
    """
    Compute empirical coverage within q-quantile bins of `values`.
    Robust to NumPy arrays; always uses pandas.Series for bin labels.
    """
    vals = pd.Series(values, copy=False).astype(float)

    # qcut can drop bins if there are many ties; allow duplicates=drop
    labels_cat = pd.qcut(vals, q=q, duplicates="drop")

    # Use a Series (not ndarray) so .unique() is available; stringify labels for CSV
    group = pd.Series(labels_cat.astype(str), index=vals.index)

    # Precompute CIs for each level
    cis = {}
    for lvl in levels:
        alpha = (1.0 - lvl) / 2.0
        lo = norm.ppf(alpha, loc=mean, scale=sd)
        hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
        cis[lvl] = (lo, hi)

    # If categorical, preserve bin order; otherwise fall back to unique() order
    try:
        ordered_bins = list(labels_cat.cat.categories.astype(str))
    except Exception:
        ordered_bins = list(group.unique())

    rows = []
    for lbl in ordered_bins:
        mask = (group.values == lbl)
        if not np.any(mask):
            continue
        mu_mean = float(np.mean(mu[mask]))
        k_mean  = float(np.mean(kappa[mask]))
        for lvl in levels:
            lo, hi = cis[lvl]
            inside = (y >= lo) & (y <= hi)
            emp = float(np.mean(inside[mask]))
            rows.append({
                "bin": str(lbl),
                "nominal": float(lvl),
                "empirical": emp,
                "bias": emp - float(lvl),
                "N_bin": int(mask.sum()),
                "mu_mean": mu_mean,
                "kappa_mean": k_mean
            })
    return pd.DataFrame(rows)


cov_by_mu   = _coverage_by_quantiles(mu,   levels=COVERAGE_LEVELS, q=10)
cov_by_l10k = _coverage_by_quantiles(l10k, levels=COVERAGE_LEVELS, q=10)
_save_csv(cov_by_mu,   _fname("coverage_by_mu_quantiles"))
_save_csv(cov_by_l10k, _fname("coverage_by_l10k_quantiles"))

print("\n Normal-approx priors diagnostics complete. All outputs written as CSVs in 'metric'.")


[outdir] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic
[loaded] 247,614 rows
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_row_metrics_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_coverage_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_coverage_grid_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_pit_uniformity_bins_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_pit_tests_normalapprox.csv


c:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\.venv\Lib\site-packages\scipy\stats\_hypotests.py:395: RuntimeWarning: overflow encountered in scalar multiply
  e3 = 2 * (m + 2) * gamma(k + 3/2) * _ed3((4 * k + 5) / sx) / (12 * y2)
c:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\.venv\Lib\site-packages\scipy\stats\_hypotests.py:393: RuntimeWarning: overflow encountered in scalar multiply
  e1 = m * gamma(k + 1/2) * _ed2((4 * k + 3)/sx) / (9 * y1)
c:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\.venv\Lib\site-packages\scipy\stats\_hypotests.py:396: RuntimeWarning: overflow encountered in scalar multiply
  e4 = 7 * m * gamma(k + 1/2) * _ed2((4 * k + 1) / sx) / (144 * y1)
c:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\.venv\Lib\site-packages\scipy\stats\_hypotests.py:397: RuntimeWarning: overflow encountered in scalar multiply
  e5 = 7 * m * gamma(k + 1/2) * _ed2((4 * k + 5) / sx) / (144 * y1)
c:\Users\osoom\OneD

[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_fit_diagnostics_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_summary_normalapprox.csv
✅ Summary written -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_summary_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_variable_summaries_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_mu_kappa_correlation_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n3ciuhehc34cy43finaltreyw1ic\priors_y_mean_correlation_normalapprox.csv
[csv] C:\Use

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

### Done
All PNGs saved under `results/priors/figures`, and all CSV metrics saved under `results/priors/metric`.